# Causal evaluation of the pre-trained SurvivEHR foundation model


In this notebook we evaluate the ability of SurvivEHR to perform next-event prediction after pre-training.

In [2]:
import os
from pathlib import Path
import sys
node_type = os.getenv('BB_CPU')
venv_dir = f'/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-{node_type}'
venv_site_pkgs = Path(venv_dir) / 'lib' / f'python{sys.version_info.major}.{sys.version_info.minor}' / 'site-packages'
if venv_site_pkgs.exists():
    sys.path.insert(0, str(venv_site_pkgs))
    print(f"Added path '{venv_site_pkgs}' at start of search paths.")
else:
    print(f"Path '{venv_site_pkgs}' not found. Check that it exists and/or that it exists for node-type '{node_type}'.")


Added path '/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-icelake/lib/python3.10/site-packages' at start of search paths.


In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import wandb
from hydra import compose, initialize
import polars as pl
import logging
import torch
import warnings
pl.Config.set_tbl_rows(10000)
logging.basicConfig(level=logging.INFO)
torch.manual_seed(1337)
torch.set_float32_matmul_precision('medium')
logging.getLogger().setLevel(logging.ERROR)          # Remove deprecation warnings from older dataset format

from FastEHR.dataloader.foundational_loader import FoundationalDataModule

from SurvivEHR.examples.modelling.SurvivEHR.run_experiment import run
from SurvivEHR.src.models.survival.task_heads.causal import SurvStreamGPTForCausalModelling

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}.")

 # TODO: define an env variable to fix for a local hpc environment issue, this shouldn't be needed
%env SLURM_NTASKS_PER_NODE=28   

Using device: cuda.
env: SLURM_NTASKS_PER_NODE=28


## Choosing configurations
The default configuration is for pre-training. Here we modify as necesssary

Here we choose to load in the configuration for our **pre-trained** model. We specfiy the `zero-shot` experiment type, which will lead to running a ```CausalExperiment```. 

We tell this experiment that no further training is needed. Additionally, we do choose to perform testing (true by default).

For demonstration, we reduce the number of test batches used to be a small fraction of those available to us. Calculating the inter-event concordance is an expensive operation due the the volume of self-supervised examples.

In [5]:
pre_trained_model, config_name = "SurvivEHR-cr-small-debug7_exp1000-v1-v4-v1", "config_CompetingRisk11M"
print(pre_trained_model)

SurvivEHR-cr-small-debug7_exp1000-v1-v4-v1


In [ ]:
wandb.finish()

# load the configuration file, override any settings 
with initialize(version_base=None, config_path="../../../../confs", job_name="causal_metric_testing_notebook"):
    cfg = compose(config_name=config_name, 
                  overrides=[# Experiment setup
                             "experiment.project_name='Evaluating pre-trained models'",
                             f"experiment.run_id='{pre_trained_model}'",
                             "experiment.train=False",
                             "experiment.test=True",
                             "data.batch_size=128",
                             "data.meta_information_path=/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/PreTrain/meta_information_QuantJenny.pickle",
                             "data.min_workers=12",
                             "optim.limit_test_batches=0.0035",
                            ]
                 )     

model, dm = run(cfg)
print(f"Loaded model with {sum(p.numel() for p in model.parameters())/1e6} M parameters")



/rds/bear-apps/2022a/EL8-ice/software/PyTorch/2.0.1-foss-2022a-CUDA-11.7.0/lib/python3.10/site-packages/torch/utils/data/dataloader.py:560: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 3, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
ERROR:wandb.jupyter:Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variabl

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.trainer.connectors.signal_connector:SLURM auto-requeueing enabled. Setting signal handlers.
/rds/bear-apps/2022a/EL8-ice/software/PyTorch/2.0.1-foss-2022a-CUDA-11.7.0/lib/python3.10/site-packages/torch/utils/data/dataloader.py:560: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 3, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


Testing: |          | 0/? [00:00<?, ?it/s]

### Extract W&B run ID and then close the session. To re-open later for plotting

In [20]:
wandb_id = wandb.run.url.split("/")[-1]
print(wandb_id)

wandb.finish()

SurvivEHR-cr-small-debug7_exp1000-v1-v4-v1
ig41w3iw
ig41w3iw


In [8]:
print(pre_trained_model)

os.makedirs(f"figs/{pre_trained_model}/", exist_ok=True) 

SurvivEHR-cr-small-debug7_exp1000-v1-v4-v1


In [22]:
wandb.login()

# Load causal_eval results from log
api = wandb.Api()

# Connect to the run
run_path = f"cwlgadd/Evaluating pre-trained models/{wandb.run.id}"     # 3omvr6q0 2qttxkwm
run = api.run(run_path) 

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


In [23]:
history = run.history(keys=None)
raw_data_from_wandb = {}
for key in history.keys():
    raw_data_from_wandb = {**raw_data_from_wandb, key: history[key].to_numpy()[-1]}
# display(raw_data_from_wandb.sort())



# Get dataloader so we can extract event names by type

In [24]:
valued_events = dm.meta_information["measurement_tables"][dm.meta_information["measurement_tables"]["count_obs"] > 0]["event"].to_list()
non_valued_events = dm.meta_information["measurement_tables"][dm.meta_information["measurement_tables"]["count_obs"] == 0]["event"].to_list()
diagnoses = dm.meta_information["diagnosis_table"]["event"].to_list()

# Next event concordance

In [25]:
Cinter_keys = [_key for _key in raw_data_from_wandb.keys() if "Test:CausalMetricssurv_stratify_by_" in _key ]

decoded_cintra_diagnoses = {}
decoded_cintra_non_valued = {}
decoded_cintra_valued = {}

for _key in Cinter_keys:
    _event = int(_key[len("Test:CausalMetricssurv_stratify_by_"):])                 # token
    _event_name = dm.decode([_event]).split(" ")[0]         # string
    _event_cintra = raw_data_from_wandb[_key]               # concordance

    if _event_name in diagnoses:# .upper() == _event_name:
        decoded_cintra_diagnoses = {**decoded_cintra_diagnoses, _event_name: _event_cintra}
    elif _event_name in non_valued_events:
        decoded_cintra_non_valued = {**decoded_cintra_non_valued, _event_name: _event_cintra}
    elif _event_name in valued_events:
        decoded_cintra_valued = {**decoded_cintra_valued, _event_name: _event_cintra}
    else:
        raise NotImplementedError


In [26]:
# display(decoded_cintra_diagnoses)
# display(decoded_cintra_other)

In [27]:
BaseCinter_keys = [_key for _key in raw_data_from_wandb.keys() if "Test:CausalMetricsprevalence_stratify_by_" in _key ]

base_decoded_cintra_diagnoses = {}
base_decoded_cintra_non_valued = {}
base_decoded_cintra_valued = {}

base_prevalence_diagnoses = {}
base_prevalence_non_valued = {}
base_prevalence_valued = {}


for _key in BaseCinter_keys:
    _event = int(_key[len("Test:CausalMetricsprevalence_stratify_by_"):])                 # token
    _event_name = dm.decode([_event]).split(" ")[0]         # string
    _event_cintra = raw_data_from_wandb[_key]               # concordance

    prevalence = dm.tokenizer._event_counts
    prevalence = prevalence.filter(pl.col("EVENT") ==_event_name)["COUNT"][0]

    if _event_name in diagnoses: #.upper() == _event_name:
        base_decoded_cintra_diagnoses = {**base_decoded_cintra_diagnoses, _event_name: _event_cintra}
        base_prevalence_diagnoses = {**base_prevalence_diagnoses, _event_name: prevalence}
        
    elif _event_name in non_valued_events:
        base_decoded_cintra_non_valued = {**base_decoded_cintra_non_valued, _event_name: _event_cintra}
        base_prevalence_non_valued = {**base_prevalence_non_valued, _event_name: prevalence}

    elif _event_name in valued_events:
        base_decoded_cintra_valued = {**base_decoded_cintra_valued, _event_name: _event_cintra}
        base_prevalence_valued = {**base_prevalence_valued, _event_name: prevalence}

    else:
        raise NotImplementedError


In [28]:
# display(base_decoded_cintra_diagnoses)

In [29]:
keys_included_diagnoses = list(set(base_decoded_cintra_diagnoses.keys()) & set(decoded_cintra_diagnoses.keys()))
keys_included_non_valued = list(set(base_decoded_cintra_non_valued.keys()) & set(decoded_cintra_non_valued.keys()))
keys_included_valued = list(set(base_decoded_cintra_valued.keys()) & set(decoded_cintra_valued.keys()))


In [30]:
for dict_name, result_dict, result_dict_base, result_dict_prev, keys_to_include in zip(["diagnoses", "medications", "measurements"],
                                                                     [decoded_cintra_diagnoses, decoded_cintra_non_valued, decoded_cintra_valued], 
                                                                     [base_decoded_cintra_diagnoses, base_decoded_cintra_non_valued, base_decoded_cintra_valued], 
                                                                     [base_prevalence_diagnoses, base_prevalence_non_valued, base_prevalence_valued],
                                                                     [keys_included_diagnoses, keys_included_non_valued, keys_included_valued]
                                                                     ):
    plt.close()
    # plt.figure(figsize=(len(keys_to_include)/5,5))
    fig, ax1 = plt.subplots(figsize=(len(keys_to_include)/4,8))
    ax2 = ax1.twinx()  

    X_axis = np.arange(len(keys_to_include)) 

    Y_base = [result_dict_base[_key] for _key in keys_to_include]
    Y_survivEHR = [result_dict[_key] for _key in keys_to_include]
    Y_log_prevalence = [np.log(result_dict_prev[_key]) for _key in keys_to_include]

    # Sort by prevalence
    arg_sort = np.argsort(Y_log_prevalence)
    Y_base = [Y_base[_i] for _i in arg_sort]
    Y_survivEHR = [Y_survivEHR[_i] for _i in arg_sort]
    Y_log_prevalence = [Y_log_prevalence[_i] for _i in arg_sort]
    keys_to_include = [keys_to_include[_i] for _i in arg_sort]

    width = 0.25
    ax1.bar(X_axis - width, Y_base, width, label = f'Concordance by prevalence (Average over events: {raw_data_from_wandb["Test:CausalMetricsprevalence_no_stratify"]:.3f})', color="mediumblue") 
    ax1.bar(X_axis, Y_survivEHR, width, label = f'Concordance by SurvivEHR (Average over events: {raw_data_from_wandb["Test:CausalMetricssurv_no_stratify"]:.3f})', color="firebrick") 
    ax2.plot(X_axis, Y_log_prevalence, width, label='Log-prevalence', color="darkseagreen", marker=".")  #  + width

    ax1.set_xticks(X_axis, keys_to_include, rotation=90) 
    # ax1.xticks(X_axis, keys_to_include) 
    ax1.set_xlabel("Events") 
    ax1.set_ylabel("Self-supervised Concordance") 
    ax2.set_ylabel("Log Prevalence") 
    ax1.legend(loc="upper left")
    ax2.legend(loc="upper right")
    ax1.set_ylim(0, 1.2)
    ax2.set_ylim(np.min(Y_log_prevalence)*0.95, np.max(Y_log_prevalence)*1.1)
    

    # plt.bar(result_dict.keys(), result_dict.values(), 0.5, color='g')
    # ax1.xticks()

    # ybar = raw_data_from_wandb["Test:Cinter"]
    # ax1.plot([0, len(result_dict)-1], 
    #          [ybar, ybar],
    #          label=f"SurvivEHR marginalised over events",
    #          color="firebrick")

    # ybar = raw_data_from_wandb["Test:base_Cinter"]
    # ax1.plot([0, len(result_dict)-1], 
    #          [ybar, ybar],
    #          label=f"Prevalence marginalised over events",
    #          color="mediumblue")
    
    plt.tight_layout()
    plt.savefig(f"figs/{pre_trained_model}/inter_causal_eval_{dict_name}.png", bbox_inches="tight")
    plt.close()

# Future events

## SurvivEHR

In [31]:
Cinter_keys = [_key for _key in raw_data_from_wandb.keys() if "+" in _key and "prevalence" not in _key ]

x_survivEHR, y_survivEHR = [], []
for _key in Cinter_keys:
    x_survivEHR.append(int(_key[6:-33]) + 1 )                # steps ahead
    y_survivEHR.append(raw_data_from_wandb[_key] )           # concordance


arg_sort = np.argsort(x_survivEHR)
x_survivEHR = [x_survivEHR[_i] for _i in arg_sort]
y_survivEHR = [y_survivEHR[_i] for _i in arg_sort]

print(x_survivEHR)
print(y_survivEHR)

[1, 2, 3, 4, 5, 8, 11, 14, 17, 20]
[0.9860266159695819, 0.8930608365019012, 0.8802768840791655, 0.8770262157294374, 0.8620616814533165, 0.8409174536980253, 0.8601303639326454, 0.7552960347637157, 0.7415856921560346, 0.7404590902689763]


In [32]:
Cinter_keys = [_key for _key in raw_data_from_wandb.keys() if "+" in _key and "prevalence" in _key ]
# print(Cinter_keys)

x_base, y_base = [], []
for _key in Cinter_keys:
    x_base.append(int(_key[6:-len("_LookAheadMetricsprevalence_no_stratify")]) + 1 )                # steps ahead
    y_base.append(raw_data_from_wandb[_key] )                   # concordance

arg_sort = np.argsort(x_base)
x_base = [x_base[_i] for _i in arg_sort]
y_base = [y_base[_i] for _i in arg_sort]

print(x_base)
print(y_base)

[1, 2, 3, 4, 5, 8, 11, 14, 17, 20]
[0.8820342205323193, 0.8820342205323193, 0.8823242663546846, 0.8826295777466481, 0.8810730882974229, 0.8688826198945174, 0.8633894622487779, 0.8633894622487779, 0.8631178707224335, 0.8631178707224335]


In [33]:
plt.close()

plt.plot(x_survivEHR, y_survivEHR,
         label=f"SurvivEHR",
         color="firebrick")

plt.plot(x_base, y_base,
         label=f"Prevalence",
         color="mediumblue")

plt.xticks(x_survivEHR)
plt.ylabel("Self-supervised multi-step concordance")
plt.xlabel("Number of steps ahead")
plt.legend()
plt.tight_layout()
plt.savefig(f"figs/{pre_trained_model}/inter_decay.png", bbox_inches="tight")
    